# AIMET-Enhanced MXFP4 Quantization for TFLite Models

This notebook applies Qualcomm AIMET optimizations before MXFP4 (Microscaling FP4) quantization:

1. **Cross-Layer Equalization (CLE)** — rebalances weight ranges across consecutive layers
2. **Bias Correction (BC)** — corrects bias shifts introduced by quantization
3. **AdaRound** — learned rounding that outperforms nearest-rounding for low-bit formats
4. **MXFP4 (E2M1)** — 4-bit microscaling quantization with block-wise E8M0 scales

**Input:** A Keras `.h5` model (or TFLite model + original Keras model)

**Output:** MXFP4-quantized `.fp4bin` file + int8 TFLite for deployment

**Why AIMET before FP4?** Going from float32 to 4 bits is aggressive. AIMET's CLE and AdaRound
prepare the weight distributions so that the 4-bit rounding loses less information.

In [ ]:
#@title 1. Install AIMET & Dependencies
# AIMET requires specific TF versions — install the compatible release
import subprocess, sys

# Install AIMET for TensorFlow (GPU build for Colab T4)
!pip install -q aimet-tensorflow-gpu 2>/dev/null || \
    pip install -q aimet-tensorflow 2>/dev/null || \
    echo "AIMET pip install failed — trying from release wheel..."

# If pip install fails, try the official release wheel
import importlib
try:
    importlib.import_module('aimet_tensorflow')
    print("AIMET TensorFlow backend installed successfully.")
except ImportError:
    print("Falling back to AIMET release wheel...")
    !pip install -q https://github.com/quic/aimet/releases/download/1.34.0/aimet_tensorflow-1.34.0-cp310-cp310-manylinux_2_34_x86_64.whl 2>/dev/null || \
        echo "Could not install AIMET wheel. Will use standalone implementations."

!pip install -q pycocotools opencv-python-headless

# Verify what we have
AIMET_AVAILABLE = False
try:
    import aimet_tensorflow
    AIMET_AVAILABLE = True
    print(f"AIMET version: {aimet_tensorflow.__version__}")
except ImportError:
    print("AIMET not available — will use standalone CLE + AdaRound implementations.")
    print("(These implement the same algorithms from the AIMET/papers directly.)")

import numpy as np
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
#@title 2. Mount Drive & Configure Paths { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')

import os

#@markdown **Path to your Keras .h5 model:**
keras_model_path = "/content/drive/MyDrive/modelVisualWakeWord.h5" #@param {type:"string"}

#@markdown **Path to calibration images (for AdaRound + TFLite int8):**
calibration_dir = "/content/drive/MyDrive/train2014" #@param {type:"string"}

#@markdown **Input image size (height = width):**
input_size = 96 #@param {type:"integer"}

#@markdown **Number of calibration samples:**
n_calibration = 200 #@param {type:"integer"}

#@markdown **MXFP4 block size (OCP default = 32):**
block_size = 32 #@param {type:"integer"}

INPUT_SHAPE = (input_size, input_size, 3)
BLOCK_SIZE = block_size

assert os.path.isfile(keras_model_path), f"Model not found: {keras_model_path}"
print(f"Model: {keras_model_path}")
print(f"Input shape: {INPUT_SHAPE}")

has_cal_data = os.path.isdir(calibration_dir)
if has_cal_data:
    n_imgs = len([f for f in os.listdir(calibration_dir) if f.lower().endswith(('.jpg','.png'))])
    print(f"Calibration images: {n_imgs} in {calibration_dir}")
else:
    print(f"No calibration dir at {calibration_dir} — will use synthetic data")

In [ ]:
#@title 3. Load Model & Build Calibration Dataset
import cv2
import numpy as np
import tensorflow as tf

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("Loading model...")
model = tf.keras.models.load_model(keras_model_path, compile=False)
model.summary()

# Build calibration dataset
def load_calibration_data(cal_dir, n_samples, img_size):
    """Load and preprocess calibration images."""
    images = []
    if cal_dir and os.path.isdir(cal_dir):
        files = sorted([f for f in os.listdir(cal_dir)
                        if f.lower().endswith(('.jpg', '.png'))])
        rng = np.random.RandomState(42)
        rng.shuffle(files)
        for fname in files[:n_samples]:
            img = cv2.imread(os.path.join(cal_dir, fname))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # Pad to square, then resize
            h, w = img.shape[:2]
            side = max(h, w)
            canvas = np.zeros((side, side, 3), dtype=img.dtype)
            y0, x0 = (side - h) // 2, (side - w) // 2
            canvas[y0:y0+h, x0:x0+w] = img
            img = cv2.resize(canvas, (img_size, img_size), interpolation=cv2.INTER_AREA)
            img = img.astype(np.float32) / 127.5 - 1.0  # MobileNet [-1, 1]
            images.append(img)
        print(f"Loaded {len(images)} calibration images from {cal_dir}")
    else:
        # Synthetic calibration data with realistic statistics
        print("Using synthetic calibration data")
        rng = np.random.RandomState(42)
        mean = np.array([-0.030, -0.088, -0.188], dtype=np.float32)
        std = np.array([0.458, 0.448, 0.450], dtype=np.float32)
        for _ in range(n_samples):
            img = rng.randn(img_size, img_size, 3).astype(np.float32) * std + mean
            img = np.clip(img, -1.0, 1.0)
            images.append(img)

    return np.array(images)

cal_data = load_calibration_data(calibration_dir, n_calibration, input_size)
print(f"Calibration data shape: {cal_data.shape}")

In [ ]:
#@title 4. MXFP4 Core — E2M1 Codec & Microscaling
import struct, gzip

# ─── FP4 E2M1 lookup table ───────────────────────────────────────────────
FP4_TABLE = np.zeros(16, dtype=np.float32)
for code in range(16):
    sign = -1.0 if (code >> 3) & 1 else 1.0
    exp_bits = (code >> 1) & 0x3
    mant_bit = code & 0x1
    if exp_bits == 0:  # subnormal
        value = 0.5 * mant_bit
    else:
        value = 2.0 ** (exp_bits - 1) * (1.0 + 0.5 * mant_bit)
    FP4_TABLE[code] = sign * value

_POS_VALUES = FP4_TABLE[:8]

def float_to_fp4(x):
    sign = 0
    if x < 0:
        sign = 8
        x = -x
    idx = int(np.argmin(np.abs(_POS_VALUES - x)))
    return sign | idx

float_to_fp4_vec = np.vectorize(float_to_fp4, otypes=[np.uint8])

def fp4_to_float(codes):
    return FP4_TABLE[codes]

def quantize_block_mx_fp4(block):
    amax = np.max(np.abs(block))
    if amax == 0:
        return 1.0, np.zeros(len(block), dtype=np.uint8)
    raw_exp = np.ceil(np.log2(amax / 6.0))
    scale = 2.0 ** raw_exp
    scaled = block / scale
    codes = float_to_fp4_vec(scaled)
    return scale, codes

def dequantize_block_mx_fp4(scale, codes):
    return scale * fp4_to_float(codes)

def pack_fp4(codes):
    n = len(codes)
    if n % 2 != 0:
        codes = np.append(codes, 0)
    packed = (codes[0::2] << 4) | codes[1::2]
    return packed.astype(np.uint8).tobytes()

def unpack_fp4(data, count):
    arr = np.frombuffer(data, dtype=np.uint8)
    hi = (arr >> 4) & 0xF
    lo = arr & 0xF
    codes = np.empty(len(arr) * 2, dtype=np.uint8)
    codes[0::2] = hi
    codes[1::2] = lo
    return codes[:count]

SKIP_KEYWORDS = {'bias', 'gamma', 'beta', 'moving_mean', 'moving_variance'}

def should_quantize(name):
    return all(kw not in name.lower() for kw in SKIP_KEYWORDS)

print("FP4 E2M1 representable values (positive):", _POS_VALUES)
print("MXFP4 codec ready.")

In [ ]:
#@title 5. Cross-Layer Equalization (CLE)
#
# CLE rebalances weight ranges between consecutive Conv/DepthwiseConv layers
# so that no single layer has an outlier range that hurts low-bit quantization.
#
# Algorithm (from "Data-Free Quantization Through Weight Equalization", Nagel et al. 2019):
#   For consecutive layers with weights W1 (out) and W2 (in):
#     scale_i = sqrt(max|W1[i,:]|  / max|W2[:,i]|)
#     W1[i,:] /= scale_i
#     W2[:,i] *= scale_i
#   This preserves the mathematical equivalence while balancing ranges.

def cross_layer_equalization(model):
    """Apply CLE to consecutive conv layers in a Keras model."""

    if AIMET_AVAILABLE:
        # Use AIMET's optimized implementation
        from aimet_tensorflow.keras.cross_layer_equalization import equalize_model
        print("Applying AIMET Cross-Layer Equalization...")
        equalize_model(model, input_shape=INPUT_SHAPE)
        print("AIMET CLE complete.")
        return model

    # ─── Standalone CLE implementation ────────────────────────────────────
    print("Applying standalone Cross-Layer Equalization...")

    # Find consecutive conv layer pairs
    conv_layers = []
    for layer in model.layers:
        if isinstance(layer, (tf.keras.layers.Conv2D,
                              tf.keras.layers.DepthwiseConv2D,
                              tf.keras.layers.Dense)):
            conv_layers.append(layer)

    n_equalized = 0
    for i in range(len(conv_layers) - 1):
        layer1 = conv_layers[i]
        layer2 = conv_layers[i + 1]

        # Skip DepthwiseConv — CLE applies to pointwise/regular conv pairs
        if isinstance(layer1, tf.keras.layers.DepthwiseConv2D):
            continue
        if isinstance(layer2, tf.keras.layers.DepthwiseConv2D):
            continue

        w1 = layer1.get_weights()
        w2 = layer2.get_weights()
        if len(w1) == 0 or len(w2) == 0:
            continue

        kernel1 = w1[0]  # shape: (..., C_out)
        kernel2 = w2[0]  # shape: (C_in, ...) or (kH, kW, C_in, C_out)

        # Get output channels of layer1 = input channels of layer2
        c_out1 = kernel1.shape[-1]

        # For Conv2D: kernel shape is (kH, kW, C_in, C_out)
        if len(kernel2.shape) == 4:
            c_in2 = kernel2.shape[2]
        elif len(kernel2.shape) == 2:
            c_in2 = kernel2.shape[0]
        else:
            continue

        if c_out1 != c_in2:
            continue  # Skip if channels don't match (e.g., skip connections)

        # Compute per-channel scale factors
        # Reshape kernels to (C, -1) for per-channel range computation
        k1_flat = kernel1.reshape(-1, c_out1)  # (spatial*C_in, C_out)
        if len(kernel2.shape) == 4:
            # (kH, kW, C_in, C_out) → (C_in, kH*kW*C_out)
            k2_flat = kernel2.transpose(2, 0, 1, 3).reshape(c_in2, -1)
        else:
            k2_flat = kernel2  # (C_in, C_out)

        range1 = np.max(np.abs(k1_flat), axis=0) + 1e-10  # (C_out,)
        range2 = np.max(np.abs(k2_flat), axis=1) + 1e-10  # (C_in,)

        # Equalization scale: S_i = sqrt(range1_i / range2_i)
        scale = np.sqrt(range1 / range2)
        scale = np.clip(scale, 0.01, 100.0)  # Prevent extreme scales

        # Apply: W1 /= S, W2 *= S
        kernel1_new = kernel1 / scale.reshape([1] * (len(kernel1.shape) - 1) + [-1])
        if len(kernel2.shape) == 4:
            kernel2_new = kernel2 * scale.reshape(1, 1, -1, 1)
        else:
            kernel2_new = kernel2 * scale.reshape(-1, 1)

        # Also adjust bias of layer1 if present
        w1_new = [kernel1_new] + ([w1[1] / scale] if len(w1) > 1 else [])
        w2_new = [kernel2_new] + list(w2[1:])

        layer1.set_weights(w1_new)
        layer2.set_weights(w2_new)
        n_equalized += 1
        print(f"  Equalized: {layer1.name} → {layer2.name}  "
              f"(scale range: [{scale.min():.3f}, {scale.max():.3f}])")

    print(f"CLE complete: {n_equalized} layer pairs equalized.")
    return model

model_cle = tf.keras.models.load_model(keras_model_path, compile=False)
model_cle = cross_layer_equalization(model_cle)

In [ ]:
#@title 6. Bias Correction
#
# Quantization introduces a systematic bias (E[W_q] != E[W]).
# Bias correction absorbs this shift into the layer's bias term.
# From: Nagel et al., "Data-Free Quantization Through Weight Equalization" (2019)

def bias_correction(model, cal_data):
    """Correct quantization-induced bias shifts using calibration data."""

    if AIMET_AVAILABLE:
        from aimet_tensorflow.keras.bias_correction import BiasCorrection
        print("Applying AIMET Bias Correction...")
        # AIMET needs a data loader callback
        def data_loader():
            for i in range(0, len(cal_data), 32):
                yield cal_data[i:i+32]
        BiasCorrection.correct_bias(model, data_loader)
        print("AIMET Bias Correction complete.")
        return model

    # ─── Standalone bias correction ───────────────────────────────────────
    print("Applying standalone Bias Correction...")

    n_corrected = 0
    for layer in model.layers:
        if not isinstance(layer, (tf.keras.layers.Conv2D,
                                  tf.keras.layers.DepthwiseConv2D)):
            continue
        if not layer.use_bias:
            continue

        weights = layer.get_weights()
        kernel = weights[0]
        bias = weights[1]

        # Simulate MXFP4 quantization of this kernel
        flat = kernel.flatten()
        pad_len = (BLOCK_SIZE - len(flat) % BLOCK_SIZE) % BLOCK_SIZE
        padded = np.concatenate([flat, np.zeros(pad_len, dtype=np.float32)])

        q_flat = np.empty_like(padded)
        for b in range(len(padded) // BLOCK_SIZE):
            blk = padded[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE]
            scale, codes = quantize_block_mx_fp4(blk)
            q_flat[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE] = dequantize_block_mx_fp4(scale, codes)

        q_kernel = q_flat[:len(flat)].reshape(kernel.shape)

        # Compute expected output difference: E[conv(x, W)] - E[conv(x, W_q)]
        # Approximate by running calibration data through a single-layer model
        # Simplified: correction = sum over spatial/input dims of (W - W_q)
        diff = kernel - q_kernel

        if isinstance(layer, tf.keras.layers.DepthwiseConv2D):
            # DepthwiseConv: kernel shape (kH, kW, C_in, 1)
            bias_correction = np.sum(diff, axis=(0, 1, 3)).flatten()
        else:
            # Conv2D: kernel shape (kH, kW, C_in, C_out)
            bias_correction = np.sum(diff, axis=(0, 1, 2))

        # Scale correction by average input magnitude (from calibration data)
        avg_input_mag = np.mean(np.abs(cal_data))
        bias_new = bias + bias_correction * avg_input_mag

        layer.set_weights([kernel, bias_new])
        correction_mag = np.mean(np.abs(bias_correction * avg_input_mag))
        if correction_mag > 1e-6:
            print(f"  Corrected {layer.name}: avg bias shift = {correction_mag:.6f}")
            n_corrected += 1

    print(f"Bias correction complete: {n_corrected} layers corrected.")
    return model

model_cle = bias_correction(model_cle, cal_data)

In [ ]:
#@title 7. AdaRound — Adaptive Rounding
#
# Instead of nearest-rounding, AdaRound learns whether to round up or down
# for each weight, minimizing the layer-wise output reconstruction error.
# From: Nagel et al., "Up or Down? Adaptive Rounding for Post-Training Quantization" (ICML 2020)
#
# For each weight w_i, we learn a continuous variable h_i ∈ [0,1] via:
#   w_q_i = floor(w_scaled_i) + h(sigmoid(v_i))
# where h is the rectified sigmoid and v_i is learned by minimizing:
#   L = ||Wx - W_q x||^2 + λ * Σ h(1-h)   (regularizer pushes h → 0 or 1)

def adaround_layer(layer, cal_inputs, n_iter=500, lr=1e-3):
    """Apply AdaRound to a single conv/dense layer.

    Args:
        layer: Keras layer with kernel weights
        cal_inputs: calibration activations at this layer's input (N, H, W, C)
        n_iter: optimization iterations
        lr: learning rate for the rounding variable

    Returns:
        Optimized kernel with adaptive rounding applied.
    """
    weights = layer.get_weights()
    kernel = weights[0].astype(np.float32)
    flat = kernel.flatten()

    # Compute per-block FP4 scales (these are fixed — we only optimize rounding)
    pad_len = (BLOCK_SIZE - len(flat) % BLOCK_SIZE) % BLOCK_SIZE
    padded = np.concatenate([flat, np.zeros(pad_len, dtype=np.float32)])
    n_blocks = len(padded) // BLOCK_SIZE

    scales = []
    for b in range(n_blocks):
        blk = padded[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE]
        amax = np.max(np.abs(blk))
        if amax == 0:
            scales.append(1.0)
        else:
            scales.append(2.0 ** np.ceil(np.log2(amax / 6.0)))
    scales = np.array(scales, dtype=np.float32)

    # Build per-element scale array
    elem_scales = np.repeat(scales, BLOCK_SIZE)[:len(padded)]

    # Scaled weights (in FP4 representable range)
    scaled_w = padded / elem_scales

    # For each scaled weight, find the two nearest FP4 values (floor & ceil)
    fp4_pos = _POS_VALUES.copy()  # [0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]
    all_fp4 = np.concatenate([-fp4_pos[::-1], fp4_pos])  # sorted FP4 values
    all_fp4 = np.unique(all_fp4)

    floor_vals = np.zeros_like(scaled_w)
    ceil_vals = np.zeros_like(scaled_w)
    for i, sw in enumerate(scaled_w):
        diffs = all_fp4 - sw
        below = all_fp4[diffs <= 0]
        above = all_fp4[diffs >= 0]
        floor_vals[i] = below[-1] if len(below) > 0 else all_fp4[0]
        ceil_vals[i] = above[0] if len(above) > 0 else all_fp4[-1]

    # If floor == ceil, no rounding decision needed
    needs_rounding = (floor_vals != ceil_vals)

    if not np.any(needs_rounding):
        return kernel  # All weights land exactly on FP4 values

    # Initialize rounding variable v: sigmoid(v) = 0.5 → start at v=0
    v = tf.Variable(tf.zeros(int(np.sum(needs_rounding)), dtype=tf.float32))
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    # Precompute the reference output using a few calibration samples
    n_cal = min(32, len(cal_inputs))
    cal_batch = cal_inputs[:n_cal]

    # Build a mini forward function for this layer
    if isinstance(layer, tf.keras.layers.Conv2D):
        conv_fn = lambda k: tf.nn.conv2d(
            cal_batch, k,
            strides=layer.strides, padding=layer.padding.upper(),
            dilations=layer.dilation_rate)
    elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
        conv_fn = lambda k: tf.nn.depthwise_conv2d(
            cal_batch, k,
            strides=[1] + list(layer.strides) + [1],
            padding=layer.padding.upper(),
            dilations=layer.dilation_rate)
    elif isinstance(layer, tf.keras.layers.Dense):
        conv_fn = lambda k: tf.matmul(cal_batch, k)
    else:
        return kernel

    ref_output = conv_fn(tf.constant(kernel))

    # Indices of weights that need rounding
    round_idx = np.where(needs_rounding)[0]
    floor_r = tf.constant(floor_vals[round_idx], dtype=tf.float32)
    ceil_r = tf.constant(ceil_vals[round_idx], dtype=tf.float32)
    elem_scales_r = tf.constant(elem_scales[round_idx], dtype=tf.float32)
    fixed_q = floor_vals.copy()
    fixed_q[~needs_rounding] = floor_vals[~needs_rounding]  # exact FP4 values
    fixed_q_scaled = (fixed_q * elem_scales)[:len(flat)]

    # Optimization loop
    beta_range = (20.0, 2.0)  # annealing: start warm, end cold
    lam = 0.01  # regularization strength

    best_loss = float('inf')
    best_v = None

    for step in range(n_iter):
        # Anneal the rectified sigmoid temperature
        progress = step / max(n_iter - 1, 1)
        beta = beta_range[0] + progress * (beta_range[1] - beta_range[0])

        with tf.GradientTape() as tape:
            # Soft rounding: h = clamp(sigmoid(v) * (1+2*eps) - eps, 0, 1)
            h = tf.sigmoid(beta * v)
            h = tf.clip_by_value(h, 0.0, 1.0)

            # Quantized values for rounding positions
            q_r = (floor_r + h * (ceil_r - floor_r)) * elem_scales_r

            # Reconstruct full quantized weight vector
            q_full = tf.Variable(tf.constant(fixed_q_scaled, dtype=tf.float32))
            indices = tf.constant(round_idx.astype(np.int32).reshape(-1, 1))
            q_all = tf.tensor_scatter_nd_update(
                tf.constant(fixed_q_scaled, dtype=tf.float32), indices, q_r)

            # Reshape to kernel shape and compute output
            q_kernel = tf.reshape(q_all, kernel.shape)
            q_output = conv_fn(q_kernel)

            # Loss = reconstruction error + regularization
            rec_loss = tf.reduce_mean((ref_output - q_output) ** 2)
            reg_loss = lam * tf.reduce_mean(h * (1.0 - h))  # push h to 0 or 1
            loss = rec_loss + reg_loss

        grads = tape.gradient(loss, [v])
        if grads[0] is not None:
            optimizer.apply_gradients(zip(grads, [v]))

        if loss.numpy() < best_loss:
            best_loss = loss.numpy()
            best_v = v.numpy().copy()

    # Final hard rounding decision
    h_final = (1.0 / (1.0 + np.exp(-2.0 * best_v))) > 0.5  # True = round up

    # Build final quantized kernel
    q_final = fixed_q.copy()
    q_final[round_idx[h_final]] = ceil_vals[round_idx[h_final]]
    q_final[round_idx[~h_final]] = floor_vals[round_idx[~h_final]]
    q_final_scaled = (q_final * elem_scales)[:len(flat)]

    pct_up = np.mean(h_final) * 100
    return q_final_scaled.reshape(kernel.shape)


def apply_adaround(model, cal_data, n_iter=500):
    """Apply AdaRound to all conv/dense layers in the model."""

    if AIMET_AVAILABLE:
        from aimet_tensorflow.keras.adaround_weight import Adaround, AdaroundParameters
        print("Applying AIMET AdaRound...")
        def data_loader():
            for i in range(0, len(cal_data), 32):
                yield cal_data[i:i+32]
        params = AdaroundParameters(data_loader, num_batches=len(cal_data)//32)
        Adaround.apply_adaround(model, params)
        print("AIMET AdaRound complete.")
        return model

    # ─── Standalone AdaRound ──────────────────────────────────────────────
    print("Applying standalone AdaRound (learned rounding for MXFP4)...")
    print(f"  Iterations per layer: {n_iter}")

    # Build intermediate activation extractor
    target_layers = []
    for layer in model.layers:
        if isinstance(layer, (tf.keras.layers.Conv2D,
                              tf.keras.layers.DepthwiseConv2D,
                              tf.keras.layers.Dense)):
            if should_quantize(layer.name):
                target_layers.append(layer)

    # For each target layer, get its input activations from calibration data
    for i, layer in enumerate(target_layers):
        print(f"  [{i+1}/{len(target_layers)}] AdaRound: {layer.name} ", end="")

        # Build a sub-model from input to this layer's input
        try:
            # Get the input tensor to this layer
            if hasattr(layer, 'input') and layer.input is not None:
                input_extractor = tf.keras.Model(
                    inputs=model.input,
                    outputs=layer.input)
                layer_inputs = input_extractor.predict(cal_data, batch_size=32, verbose=0)
            else:
                # Fallback: use raw calibration data (only valid for first layer)
                layer_inputs = cal_data
        except Exception:
            print("(skipped — could not extract activations)")
            continue

        # Run AdaRound optimization
        optimized_kernel = adaround_layer(layer, layer_inputs, n_iter=n_iter)

        # Set the optimized kernel back
        w = layer.get_weights()
        w[0] = optimized_kernel
        layer.set_weights(w)

        # Report improvement
        orig_kernel = layer.get_weights()[0]
        print(f"shape={orig_kernel.shape}")

    print(f"AdaRound complete: {len(target_layers)} layers optimized.")
    return model

model_ada = apply_adaround(model_cle, cal_data, n_iter=500)

In [ ]:
#@title 8. MXFP4 Quantize — Baseline vs AIMET-Enhanced

def quantize_model_to_mxfp4(model, label=""):
    """Quantize all kernel weights to MXFP4. Returns entries list."""
    print(f"\nQuantizing to MXFP4 [{label}]...")
    entries = []
    total_orig = 0
    total_fp4 = 0
    total_kept = 0

    for idx, w in enumerate(model.weights):
        name = f"w{idx}_{w.name}"
        arr = w.numpy().flatten()
        shape = w.shape
        total_orig += arr.nbytes

        if should_quantize(w.name):
            pad_len = (BLOCK_SIZE - len(arr) % BLOCK_SIZE) % BLOCK_SIZE
            padded = np.concatenate([arr, np.zeros(pad_len, dtype=np.float32)])
            n_blocks = len(padded) // BLOCK_SIZE

            all_scales, all_codes = [], []
            for b in range(n_blocks):
                block = padded[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE]
                scale, codes = quantize_block_mx_fp4(block)
                all_scales.append(scale)
                all_codes.append(codes)

            scales = np.array(all_scales, dtype=np.float32)
            codes = np.concatenate(all_codes)[:len(arr)]
            fp4_size = scales.nbytes + (len(arr) + 1) // 2
            total_fp4 += fp4_size

            entries.append({
                'name': name, 'shape': list(shape),
                'n_elements': len(arr), 'quantized': True,
                'scales': scales, 'codes': codes,
            })
        else:
            total_kept += arr.nbytes
            entries.append({
                'name': name, 'shape': list(shape),
                'n_elements': len(arr), 'quantized': False,
                'raw_data': arr.astype(np.float32),
            })

    total_comp = total_fp4 + total_kept
    print(f"  Original: {total_orig:,} B | FP4 kernels: {total_fp4:,} B | "
          f"BN/bias: {total_kept:,} B | Total: {total_comp:,} B | "
          f"Ratio: {total_orig/total_comp:.1f}x")
    return entries

def compute_quantization_rmse(model, entries):
    """Compute overall RMSE of quantized vs original weights."""
    total_mse, total_n = 0, 0
    for idx, w in enumerate(model.weights):
        entry = entries[idx]
        if not entry.get('quantized', True):
            continue
        orig = w.numpy()
        n = entry['n_elements']
        scales, codes = entry['scales'], entry['codes']
        pad_len = (BLOCK_SIZE - n % BLOCK_SIZE) % BLOCK_SIZE
        codes_p = np.concatenate([codes, np.zeros(pad_len, dtype=np.uint8)])
        result = np.empty(len(codes_p), dtype=np.float32)
        for b in range(len(codes_p) // BLOCK_SIZE):
            s = scales[b]
            c = codes_p[b*BLOCK_SIZE:(b+1)*BLOCK_SIZE]
            result[b*BLOCK_SIZE:(b+1)*BLOCK_SIZE] = dequantize_block_mx_fp4(s, c)
        recon = result[:n].reshape(entry['shape'])
        diff = orig - recon
        total_mse += np.sum(diff ** 2)
        total_n += diff.size
    return np.sqrt(total_mse / total_n)

# Quantize both: baseline (no AIMET) and AIMET-enhanced
model_baseline = tf.keras.models.load_model(keras_model_path, compile=False)
entries_baseline = quantize_model_to_mxfp4(model_baseline, "Baseline (nearest rounding)")
rmse_baseline = compute_quantization_rmse(model_baseline, entries_baseline)

entries_aimet = quantize_model_to_mxfp4(model_ada, "AIMET CLE + AdaRound")
rmse_aimet = compute_quantization_rmse(model_ada, entries_aimet)

print(f"\n{'='*60}")
print(f"Quantization RMSE comparison:")
print(f"  Baseline (nearest round):  {rmse_baseline:.6f}")
print(f"  AIMET CLE + AdaRound:      {rmse_aimet:.6f}")
improvement = (1 - rmse_aimet / rmse_baseline) * 100
print(f"  Improvement:               {improvement:+.1f}%")
print(f"{'='*60}")

In [ ]:
#@title 9. Save MXFP4 Binary & Convert to int8 TFLite

MAGIC = b'MXF4'
FORMAT_VERSION = 2

def save_fp4_binary(entries, path):
    with open(path, 'wb') as f:
        f.write(MAGIC)
        f.write(struct.pack('<B', FORMAT_VERSION))
        f.write(struct.pack('<I', len(entries)))
        for e in entries:
            quantized = e.get('quantized', True)
            f.write(struct.pack('<B', 1 if quantized else 0))
            name_bytes = e['name'].encode('utf-8')
            f.write(struct.pack('<H', len(name_bytes)))
            f.write(name_bytes)
            shape = e['shape']
            f.write(struct.pack('<B', len(shape)))
            for s in shape:
                f.write(struct.pack('<I', s))
            f.write(struct.pack('<I', e['n_elements']))
            if quantized:
                scales = e['scales']
                f.write(struct.pack('<I', len(scales)))
                f.write(scales.tobytes())
                packed = pack_fp4(e['codes'])
                f.write(struct.pack('<I', len(packed)))
                f.write(packed)
            else:
                f.write(e['raw_data'].astype(np.float32).tobytes())
    return os.path.getsize(path)

def apply_fp4_weights(model, entries):
    new_weights = []
    for idx, w in enumerate(model.weights):
        entry = entries[idx]
        if not entry.get('quantized', True):
            new_weights.append(entry['raw_data'].reshape(entry['shape']))
        else:
            n = entry['n_elements']
            scales, codes = entry['scales'], entry['codes']
            pad_len = (BLOCK_SIZE - n % BLOCK_SIZE) % BLOCK_SIZE
            codes_p = np.concatenate([codes, np.zeros(pad_len, dtype=np.uint8)])
            result = np.empty(len(codes_p), dtype=np.float32)
            for b in range(len(codes_p) // BLOCK_SIZE):
                s = scales[b]
                c = codes_p[b*BLOCK_SIZE:(b+1)*BLOCK_SIZE]
                result[b*BLOCK_SIZE:(b+1)*BLOCK_SIZE] = dequantize_block_mx_fp4(s, c)
            new_weights.append(result[:n].reshape(entry['shape']))
    model.set_weights(new_weights)

# Save AIMET-enhanced version
os.makedirs('aimet_output', exist_ok=True)

fp4_path = 'aimet_output/model_aimet_mxfp4.fp4bin'
fp4_size = save_fp4_binary(entries_aimet, fp4_path)

fp4_gz = gzip.compress(open(fp4_path, 'rb').read(), compresslevel=9)
with open(fp4_path + '.gz', 'wb') as f:
    f.write(fp4_gz)

print(f"MXFP4 binary:       {fp4_size:>7,} B ({fp4_size/1024:.1f} KB)")
print(f"MXFP4 binary + gz:  {len(fp4_gz):>7,} B ({len(fp4_gz)/1024:.1f} KB)")

# Convert to int8 TFLite
print("\nConverting AIMET-enhanced FP4 model → int8 TFLite...")
model_for_tflite = tf.keras.models.load_model(keras_model_path, compile=False)
apply_fp4_weights(model_for_tflite, entries_aimet)

def rep_data():
    for i in range(min(n_calibration, len(cal_data))):
        yield [cal_data[i:i+1]]

converter = tf.lite.TFLiteConverter.from_keras_model(model_for_tflite)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_bytes = converter.convert()

tflite_path = 'aimet_output/model_aimet_mxfp4_int8.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_bytes)

tflite_gz = gzip.compress(tflite_bytes, compresslevel=9)
with open(tflite_path + '.gz', 'wb') as f:
    f.write(tflite_gz)

print(f"int8 TFLite:        {len(tflite_bytes):>7,} B ({len(tflite_bytes)/1024:.1f} KB)")
print(f"int8 TFLite + gz:   {len(tflite_gz):>7,} B ({len(tflite_gz)/1024:.1f} KB)")

In [ ]:
#@title 10. Compare All Variants — Final Summary

print(f"{'='*70}")
print("FINAL COMPARISON: Baseline MXFP4 vs AIMET-Enhanced MXFP4")
print(f"{'='*70}")

# Sizes
orig_h5_size = os.path.getsize(keras_model_path)
print(f"\n  Original Keras .h5:         {orig_h5_size:>7,} B ({orig_h5_size/1024:.1f} KB)")

rows = [
    ("Baseline MXFP4 (.fp4bin)", save_fp4_binary(entries_baseline, '/tmp/bl.fp4bin')),
    ("AIMET MXFP4 (.fp4bin)", fp4_size),
    ("AIMET MXFP4 + gzip", len(fp4_gz)),
    ("AIMET MXFP4 → int8 TFLite", len(tflite_bytes)),
    ("AIMET MXFP4 → int8 + gzip", len(tflite_gz)),
]

print(f"\n  {'Variant':<35} {'Size':>8}  {'vs Original':>12}")
print(f"  {'-'*35} {'-'*8}  {'-'*12}")
for label, size in rows:
    pct = (1 - size / orig_h5_size) * 100
    print(f"  {label:<35} {size/1024:>7.1f}K  {pct:>+11.1f}%")

print(f"\n  Weight RMSE:")
print(f"    Baseline (nearest round):  {rmse_baseline:.6f}")
print(f"    AIMET CLE + AdaRound:      {rmse_aimet:.6f}")
print(f"    Improvement:               {(1-rmse_aimet/rmse_baseline)*100:+.1f}%")
print(f"\n  NOTE: Run on COCO val for real accuracy comparison.")
print(f"{'='*70}")

In [ ]:
#@title 11. (Optional) Download Output Files
from google.colab import files

print("Available files:")
for f in os.listdir('aimet_output'):
    size = os.path.getsize(os.path.join('aimet_output', f))
    print(f"  {f}: {size:,} B ({size/1024:.1f} KB)")

# Uncomment to download:
# files.download('aimet_output/model_aimet_mxfp4.fp4bin')
# files.download('aimet_output/model_aimet_mxfp4_int8.tflite')